# Credit Card Fraud Detection

**Goal:** Detect fraudulent transactions from highly imbalanced data
**Algorithm:** Random Forest + Undersampling

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, precision_recall_curve)

In [ ]:
np.random.seed(42)
n = 5000
# Simulate 28 anonymized features (V1-V28) + Amount + Time
X_data = np.random.randn(n, 28)
amount = np.random.exponential(50, n)
time = np.random.randint(0, 100000, n)

# Only ~0.2% are fraud (realistic ratio for credit card data)
fraud_indices = np.random.choice(n, size=int(n * 0.002), replace=False)
y = np.zeros(n)
y[fraud_indices] = 1

X_df = pd.DataFrame(X_data, columns=['V%d' % i for i in range(1, 29)])
X_df['Amount'] = amount
X_df['Time'] = time

print ('Shape: %s' % (X_df.shape,))
print ('Fraud cases: %d (%.4f%%)' % (y.sum(), y.mean() * 100))

<hr>## 1. Check Class Imbalance

In [ ]:
print ('Class distribution:')
print ('Normal transactions: %d' % (len(y) - y.sum()))
print ('Fraud transactions:  %d' % y.sum())
print ('Ratio: 1 fraud per %d normal' % ((len(y) - y.sum()) // y.sum()))

<hr>## 2. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_df, y, test_size=0.3, random_state=42, stratify=y)
print ('Train: %s, Test: %s' % (X_train.shape[0], X_test.shape[0]))

<hr>## 3. Handle Imbalance (Undersampling)

In [ ]:
fraud_train = X_train[y_train == 1]
normal_train = X_train[y_train == 0]
n_fraud = len(fraud_train)

normal_sample = normal_train.sample(n=n_fraud, random_state=42)
X_balanced = pd.concat([fraud_train, normal_sample])
y_balanced = np.array([1] * n_fraud + [0] * n_fraud)

print ('Balanced dataset size: %d' % len(X_balanced))
print ('Fraud: %d, Normal: %d' % (y_balanced.sum(), len(y_balanced) - y_balanced.sum()))

<hr>## 4. Train Model

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_balanced, y_balanced)
print ('Model: %s' % model)

<hr>## 5. Evaluate Performance

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print ('ROC-AUC Score: %.4f' % roc_auc_score(y_test, y_prob))
print ('\nConfusion Matrix:\n%s' % confusion_matrix(y_test, y_pred))

In [ ]:
print ('Classification Report:\n%s' % classification_report(
    y_test, y_pred, target_names=['Normal', 'Fraud']))

<hr>## 6. Feature Importance

In [ ]:
importances = pd.DataFrame({
    'feature': X_df.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
print ('Top 5 important features:\n%s' % importances.head(5).to_string(index=False))